# Istio + Teleport Workload Identity — Interactive Demo Walkthrough

This notebook walks through the full demo step-by-step, running the exact commands from the README and explaining what each one does.

## What This Demo Proves

Standard Istio manages its own internal Certificate Authority (CA) called *Citadel*. This demo **replaces** Citadel with Teleport as the certificate issuer. The result:

- Every workload (pod) in the mesh gets a **cryptographic identity** rooted in Teleport — not a self-signed cluster-local CA.
- Istio enforces **mutual TLS (mTLS)** using those Teleport-issued certs for every service-to-service call.
- Kubernetes `AuthorizationPolicy` rules reference **SPIFFE IDs** — meaning access control is based on *who the caller cryptographically is*, not just its IP address or label.

The standard used to express identity is [SPIFFE](https://spiffe.io/) (Secure Production Identity Framework For Everyone). A SPIFFE ID looks like:
```
spiffe://your-teleport-cluster/ns/sock-shop/sa/front-end
```

---
**Run each cell in order.** Some cells have `# ACTION REQUIRED` comments where you may need to edit a value before running.

## 0. Notebook Setup

A small helper function that runs shell commands and streams their output with coloured pass/fail indicators. All subsequent cells use this instead of raw `subprocess` calls so you can see exactly what ran and whether it succeeded.

In [45]:
import subprocess
import os
import shutil
import time
from IPython.display import display, Markdown

REPO_DIR = os.path.dirname(os.path.abspath("demo-walkthrough.ipynb"))


class _RunResult:
    """Wraps CompletedProcess with a clean Jupyter display (no ugly repr)."""
    def __init__(self, proc):
        object.__setattr__(self, '_proc', proc)

    def __getattr__(self, name):
        return getattr(object.__getattribute__(self, '_proc'), name)

    def _ipython_display_(self, **kwargs):
        # stdout was already printed by run(); suppress the auto-repr
        pass


def run(cmd, *, cwd=REPO_DIR, check=True, capture=False, delay=0):
    """Run a shell command, print output, return CompletedProcess."""
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd, shell=True, cwd=cwd,
        capture_output=capture,
        text=True
    )
    if capture:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    if delay:
        time.sleep(delay)
    return _RunResult(result)

def wait_for_pods(namespace, *, timeout=120, interval=5):
    """Poll kubectl get pods until all pods in the namespace are Ready."""
    deadline = time.time() + timeout
    while True:
        result = subprocess.run(
            f"kubectl get pods -n {namespace}",
            shell=True, capture_output=True, text=True
        )
        print(result.stdout, end="")
        lines = result.stdout.strip().splitlines()
        pod_lines = lines[1:] if len(lines) > 1 else []
        def pod_ready(line):
            cols = line.split()
            if len(cols) < 3:
                return False
            status = cols[2]
            if status in ("Completed", "Terminating"):
                return True
            if "/" in cols[1]:
                current, total = cols[1].split("/", 1)
                return current == total and current != "0"
            return False
        if pod_lines and all(pod_ready(l) for l in pod_lines):
            print(f"✅ All pods ready in {namespace}")
            return
        if time.time() >= deadline:
            raise TimeoutError(f"Pods in {namespace} not ready after {timeout}s")
        print(f"  (retrying in {interval}s...)\n")
        time.sleep(interval)

print("Helper loaded. Working directory:", REPO_DIR)

Helper loaded. Working directory: /Users/jeff/dev/rev-tech/use-cases/mwi/istio-spiffe


In [46]:
for tool in ["kubectl", "istioctl", "tctl", "tsh"]:
    path = shutil.which(tool)
    status = f"✅ found at {path}" if path else "❌ NOT FOUND — install before continuing"
    print(f"{tool:12s}  {status}")

print()
run("kubectl cluster-info", capture=True)
run("istioctl version --remote=false", capture=True)
run("tctl status", capture=True)

kubectl       ✅ found at /opt/homebrew/bin/kubectl
istioctl      ✅ found at /opt/homebrew/bin/istioctl
tctl          ✅ found at /usr/local/bin/tctl
tsh           ✅ found at /usr/local/bin/tsh

$ kubectl cluster-info
Kubernetes control plane is running at https://ellinj.teleport.sh:443
CoreDNS is running at https://ellinj.teleport.sh:443/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.

$ istioctl version --remote=false
client version: 1.28.0

$ tctl status
Cluster: ellinj.teleport.sh                                                      
Version: 18.8.0                                                                  
CA pins: sha256:485fa3809aa1d415a638b23cb5baa9f5562df70332da93da67619b6a87d9351e 

authority     rotation                protocol status algorithm   storage 
------------- ----------------------- -------- ------ ----------- ------- 
host          standby (never rotated) SSH      acti

## Step 1 — Configure the Teleport Trust Domain

> **Before running this cell:** copy `.env.example` to `.env` and set your cluster domain:
> ```
> cp .env.example .env
> # edit .env and set TELEPORT_TRUST_DOMAIN=your-cluster.teleport.sh
> ```

Every certificate in this demo is anchored to your **Teleport cluster domain** (e.g. `example.teleport.sh`). This domain must be consistent across:

- The Istio mesh config (`trustDomain` field) — so Istio knows which SPIFFE root to trust.
- The tbot config (`proxy_server`) — so tbot knows which Teleport cluster to authenticate against.
- The tbot DaemonSet (`audience` on the projected token) — so the Kubernetes join token targets the right cluster.
- The Istio `AuthorizationPolicy` principals — so policy rules reference valid SPIFFE IDs.

The `configure-trust-domain.sh` script reads `TELEPORT_TRUST_DOMAIN` from `.env` and does a find-and-replace across all four files in one shot.

In [47]:
env_path = os.path.join(REPO_DIR, ".env")
if not os.path.exists(env_path):
    raise FileNotFoundError(
        ".env not found — copy .env.example and set TELEPORT_TRUST_DOMAIN:\n"
        "  cp .env.example .env"
    )

# Parse .env manually (no external deps)
env_vars = {}
with open(env_path) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            env_vars[k.strip()] = v.strip()

TELEPORT_TRUST_DOMAIN = env_vars.get("TELEPORT_TRUST_DOMAIN", "")
if not TELEPORT_TRUST_DOMAIN or TELEPORT_TRUST_DOMAIN == "your-cluster.teleport.sh":
    raise ValueError("TELEPORT_TRUST_DOMAIN is not set in .env — edit the file before continuing")

print(f"Trust domain: {TELEPORT_TRUST_DOMAIN}")

# Run the configure script — it patches all YAML files in-place
run("./configure-trust-domain.sh", capture=True)

# Confirm the trust domain landed in the Istio config
run("grep trustDomain istio/istio-config.yaml", capture=True)

Trust domain: ellinj.teleport.sh
$ ./configure-trust-domain.sh
Configuring trust domain: ellinj.teleport.sh

⊘ Already configured: istio/istio-config.yaml
⊘ Already configured: tbot/tbot-config.yaml
⊘ Already configured: tbot/tbot-daemonset.yaml
⊘ Already configured: sockshop/sock-shop-policies.yaml

All files are already configured with a trust domain.
If you want to change it, manually edit the files or reset them first.

$ grep trustDomain istio/istio-config.yaml
    trustDomain: ellinj.teleport.sh



## Step 0 — Verify Prerequisites

Before touching the cluster we confirm that the three required CLI tools are present and reachable:

| Tool | Why it's needed |
|------|----------------|
| `kubectl` | Applies Kubernetes manifests and inspects pod/service state |
| `istioctl` | Installs Istio into the cluster and queries mesh state |
| `tctl` / `tsh` | Manages Teleport resources (tokens, roles, workload identities) |

`kubectl cluster-info` also confirms you're pointing at the right cluster — double-check the API server URL before proceeding.

## Step 2 — Install Istio with SPIFFE Integration

The `istio-install.sh` script runs `istioctl install` using [istio/istio-config.yaml](istio/istio-config.yaml). Key settings:

```yaml
meshConfig:
  trustDomain: <your-teleport-cluster>   # Root of all SPIFFE IDs in the mesh
  pathNormalization:
    normalization: NONE                  # Prevents trailing-slash rewrites that
                                         # break SPIFFE ID matching in AuthorizationPolicy
```

```yaml
values:
  sidecarInjectorWebhook:
    defaultTemplates: [sidecar, tbot-socket]
    templates:
      tbot-socket: |
        spec:
          volumes:
          - name: workload-socket
            hostPath:
              path: /run/spire/sockets   # tbot DaemonSet socket directory
              type: DirectoryOrCreate
```

The default `sidecar` injection template defines `workload-socket` as `emptyDir`. The
`tbot-socket` template is merged after it, replacing the emptyDir with a hostPath to
the tbot DaemonSet socket directory. pilot-agent then auto-detects the socket at
`/var/run/secrets/workload-spiffe-uds/socket` and uses the SPIFFE Workload API
(not the Citadel CA client) to fetch SVIDs issued by Teleport.

After installation `istio-system` pods should all reach `Running` status. This typically takes ~60 seconds.

In [49]:
run("./istio-install.sh", capture=True)

print("\n--- Istio system pods ---")
run("kubectl get pods -n istio-system", capture=True)

$ ./istio-install.sh
=== Istio Installation Script ===

Using istioctl version: client version: 1.28.0

Current cluster: ellinj.teleport.sh-prod-cluster

=== Creating istio-system namespace ===
namespace/istio-system unchanged
=== Installing Istio with SPIFFE integration ===

=== Waiting for Istio components to be ready ===
deployment.apps/istiod condition met

=== Istio Installation Complete ===

Verifying installation:
NAME                                    READY   STATUS        RESTARTS   AGE
istio-ingressgateway-78cd5dc979-vxqfq   0/1     Terminating   0          15m
istio-ingressgateway-7f46ff45-92dft     1/1     Running       0          4s
istiod-54b8699b8d-xdhpq                 1/1     Running       0          16m

NAME                          TYPE           CLUSTER-IP       EXTERNAL-IP     PORT(S)                                      AGE
istio-ingressgateway          LoadBalancer   10.96.140.185    192.168.1.229   15021:32746/TCP,80:32119/TCP,443:32598/TCP   15m
istiod       

## Step 3 — Create Teleport Resources

Three Teleport resources must exist before tbot can join the cluster and issue identities:

### 3a. Extract cluster JWKS and generate the join token

`create-token.sh` calls the Kubernetes API to retrieve the cluster's **JSON Web Key Set (JWKS)** — the public keys Kubernetes uses to sign projected service account tokens. Teleport embeds this JWKS in the join token so it can **cryptographically verify** that the token presented by tbot was actually minted by this specific cluster. This is how tbot joins Teleport without a password or static secret.

The result is written to `istio/istio-tbot-token.yaml`.

In [50]:
run("./create-token.sh", capture=True)

# Confirm the file was created (it's gitignored because it contains cluster-specific keys)
token_file = os.path.join(REPO_DIR, "istio", "istio-tbot-token.yaml")
if os.path.exists(token_file):
    print(f"\n✅ {token_file} generated")
else:
    print(f"\n❌ {token_file} was NOT created — check create-token.sh output above")

$ ./create-token.sh
=== Teleport Join Token Creation Script ===

Current cluster: ellinj.teleport.sh-prod-cluster

=== Extracting cluster JWKS ===
Successfully extracted JWKS

=== Creating istio/istio-tbot-token.yaml ===
Successfully created istio/istio-tbot-token.yaml

=== Next Steps ===
1. Review the generated file: cat istio/istio-tbot-token.yaml
2. Create the token in Teleport: tctl create -f istio/istio-tbot-token.yaml
3. Verify token creation: tctl get token/istio-tbot-k8s-join

IMPORTANT: istio/istio-tbot-token.yaml is gitignored and should NOT be committed
           This file contains cluster-specific secrets


✅ /Users/jeff/dev/rev-tech/use-cases/mwi/istio-spiffe/istio/istio-tbot-token.yaml generated


### 3b. Create the join token in Teleport

The token file (`istio-tbot-token.yaml`) tells Teleport:
- **Name**: `istio-tbot-k8s-join` — tbot will reference this by name.
- **Join method**: `kubernetes` — use the Kubernetes projected token mechanism.
- **JWKS**: the cluster's public signing keys (from Step 3a).
- **Rules**: which Kubernetes service accounts are allowed to use this token.

When tbot starts, it presents a short-lived Kubernetes token to Teleport. Teleport validates the token's signature against the embedded JWKS and, if valid, issues a Teleport identity certificate.

In [51]:
run("tctl create -f istio/istio-tbot-token.yaml", capture=True)

# Verify the token was created
run("tctl get token/istio-tbot-k8s-join", capture=True)

$ tctl create -f istio/istio-tbot-token.yaml
provision_token "istio-tbot-k8s-join" has been created

$ tctl get token/istio-tbot-k8s-join
kind: token
metadata:
  name: istio-tbot-k8s-join
  revision: d8fd0f77-14eb-4a7a-b259-4d7035de6e48
spec:
  bot_name: istio-tbot
  join_method: kubernetes
  kubernetes:
    allow:
    - service_account: teleport-system:tbot
    static_jwks:
      jwks: '{"keys":[{"use":"sig","kty":"RSA","kid":"u6FLatAaiV-BEJn6KS_p4AbdXET0cyIJ7T1PuBALtec","alg":"RS256","n":"zpHwGa7WrnY8sdyrBy3Bo5Jo9PziEruedY154c5wfNi98CYMkElvOU2MLizvSFa1OW6NG2nWIrGA6ev37p7cuBUwyjMJvwYmiGRfBMvmcXWOBL0a7w3gWr35uzUSDwY6NOQFyaaruyfiolxFVgTj-hgwWMtFFIh8LxbGV5YHbkTEcdgSIv0SVZertfupZcj5j4-DxZhZULZUEIsbrhRu8IrfcIIvbO9HKX4OyQj4vrvWlUJVCnu6MQ8K9aEoz3GpAc0rxD7J7tIRItAcMSfSGAB7U04UR8qpfKTf_EZSkMSBWJTvCnXzm-gjO5-S9KpTTmv9Bv5GydsJat0dcG8FUQ","e":"AQAB"}]}'
    type: static_jwks
  roles:
  - Bot
version: v2



### 3c. Create the bot role

The **bot role** (`teleport-bot-role.yaml`) grants the tbot service account just enough Teleport permissions to issue workload identity certificates. It follows the principle of least privilege — tbot can only do one thing: issue SPIFFE SVIDs (short-lived identity certificates) for the `istio-workloads` identity.

In [52]:
run("tctl create -f tbot/teleport-bot-role.yaml", capture=True,delay=3)
run("tctl get role/istio-workload-identity-issuer", capture=True)

$ tctl create -f tbot/teleport-bot-role.yaml
role "istio-workload-identity-issuer" has been created

$ tctl get role/istio-workload-identity-issuer
kind: role
metadata:
  name: istio-workload-identity-issuer
  revision: f27102c4-0a00-45d0-9db3-286be08d925f
spec:
  allow:
    rules:
    - resources:
      - workload_identity
      verbs:
      - list
      - read
    workload_identity_labels:
      env: dev
  deny: {}
  options:
    cert_format: standard
    create_db_user: false
    create_desktop_user: false
    desktop_clipboard: true
    desktop_directory_sharing: true
    enhanced_recording:
    - command
    - network
    forward_agent: false
    idp:
      saml:
        enabled: true
    max_session_ttl: 30h0m0s
    pin_source_ip: false
    record_session:
      default: best_effort
      desktop: true
    ssh_file_copy: true
version: v7



### 3d. Create the WorkloadIdentity resource

The `WorkloadIdentity` resource (`teleport-workload-identity.yaml`) defines the **template** Teleport uses to construct SPIFFE IDs:

```yaml
spec:
  spiffe:
    id: /ns/{{ workload.kubernetes.namespace }}/sa/{{ workload.kubernetes.service_account }}
```

When tbot requests an SVID for a workload, it passes Kubernetes attestation data (namespace, service account, pod name). Teleport evaluates this template and issues a certificate whose Subject Alternative Name (SAN) is:

```
spiffe://YOUR-DOMAIN/ns/<namespace>/sa/<service-account>
```

This matches exactly the SPIFFE ID format Istio expects for authorization policy principals.

In [53]:
run("tctl create -f tbot/teleport-workload-identity.yaml", capture=True, delay=3)
run("tctl get workload_identity/istio-workloads", capture=True)

$ tctl create -f tbot/teleport-workload-identity.yaml
Workload identity "istio-workloads" has been created

$ tctl get workload_identity/istio-workloads
kind: workload_identity
metadata:
  labels:
    env: dev
  name: istio-workloads
  revision: fa1dd56f-48e2-491e-9d50-995a24e86dd9
spec:
  spiffe:
    id: /ns/{{ workload.kubernetes.namespace }}/sa/{{ workload.kubernetes.service_account
      }}
version: v1



## Step 4 — Deploy tbot

tbot (Teleport Bot) is the component that bridges Kubernetes and Teleport. It runs as a **DaemonSet** — one pod per node — because Istio sidecars need to reach the SPIFFE Workload API via a **Unix domain socket on the local node filesystem**, not over the network.

How it works:

```
Istio sidecar                tbot pod (same node)
──────────────               ──────────────────────
  requests SVID  ──socket──▶  validates workload
                              via Kubelet API
                              calls Teleport
                ◀──cert──────  returns SPIFFE SVID
```

The socket lives at `/run/spire/sockets/socket` (a hostPath volume), so every sidecar on the node can reach it. This is the standard SPIFFE Workload API location — no Istio-specific configuration required.

Three manifests are applied:
| File | What it creates |
|------|-----------------|
| `tbot-rbac.yaml` | ServiceAccount, ClusterRole, ClusterRoleBinding — lets tbot call the Kubelet API to verify pod identity |
| `tbot-config.yaml` | ConfigMap with `tbot.yaml` — points tbot at the Teleport proxy and configures the workload identity service |
| `tbot-daemonset.yaml` | The DaemonSet itself — mounts the socket dir as a hostPath and injects the projected SA token |


In [54]:
run("kubectl apply -f tbot/tbot-rbac.yaml", capture=True)
run("kubectl apply -f tbot/tbot-config.yaml", capture=True)
run("kubectl apply -f tbot/tbot-daemonset.yaml", capture=True)

print("\n--- tbot pods (one per node expected) ---")
#run("kubectl get pods -n teleport-system", capture=True)
wait_for_pods("teleport-system", timeout=60)


$ kubectl apply -f tbot/tbot-rbac.yaml
namespace/teleport-system created
serviceaccount/tbot created
clusterrole.rbac.authorization.k8s.io/tbot created
clusterrolebinding.rbac.authorization.k8s.io/tbot created

$ kubectl apply -f tbot/tbot-config.yaml
configmap/tbot-config created

$ kubectl apply -f tbot/tbot-daemonset.yaml
daemonset.apps/tbot created


--- tbot pods (one per node expected) ---
NAME         READY   STATUS              RESTARTS   AGE
tbot-jnldt   0/1     ContainerCreating   0          0s
tbot-kkkcl   0/1     Pending             0          0s
tbot-q2ccm   0/1     ContainerCreating   0          0s
  (retrying in 5s...)

NAME         READY   STATUS    RESTARTS   AGE
tbot-jnldt   1/1     Running   0          5s
tbot-kkkcl   1/1     Running   0          5s
tbot-q2ccm   1/1     Running   0          5s
✅ All pods ready in teleport-system


In [55]:
# Wait for tbot to connect to Teleport and open the Workload API socket.
# This line checks the logs for the "Workload API" string — its presence means
# tbot successfully joined Teleport and is ready to serve SVIDs.
print("Waiting for tbot Workload API to become ready...")
run(
    "kubectl logs -n teleport-system -l app=tbot --tail=50 | grep -i 'Workload API'",
    capture=True
)
print("\n✅ tbot is serving the SPIFFE Workload API")

Waiting for tbot Workload API to become ready...
$ kubectl logs -n teleport-system -l app=tbot --tail=50 | grep -i 'Workload API'
2026-05-13T13:59:08.838Z INFO [TBOT:SVC:] Listener opened for Workload API endpoint addr:/run/spire/sockets/socket workloadidentity/workload_api.go:236
2026-05-13T13:59:10.545Z INFO [TBOT:SVC:] Listener opened for Workload API endpoint addr:/run/spire/sockets/socket workloadidentity/workload_api.go:236
2026-05-13T13:59:08.824Z INFO [TBOT:SVC:] Listener opened for Workload API endpoint addr:/run/spire/sockets/socket workloadidentity/workload_api.go:236


✅ tbot is serving the SPIFFE Workload API


## Step 5 — Deploy the Sock Shop Demo Application

[Sock Shop](https://github.com/microservices-demo/microservices-demo) is a canonical microservices reference app — an e-commerce site with ~10 services (front-end, catalogue, carts, orders, users, payment, etc.).

We deploy it here because it gives us **realistic service-to-service traffic** to put under Teleport/Istio policy control. Each service:
- Gets its own Kubernetes **ServiceAccount** (the source of its SPIFFE identity).
- Gets an Istio sidecar injected automatically (the `sock-shop` namespace has `istio-injection: enabled`).
- Immediately contacts tbot via the socket to request a SPIFFE SVID from Teleport.

Pods show `2/2` READY because each pod runs **two containers**: the app container + the `istio-proxy` sidecar.

In [56]:
run("kubectl apply -f sockshop/sock-shop-demo.yaml", capture=True)

print("\nWaiting for Sock Shop pods to start (this takes 1-2 minutes)...")
run(
    "kubectl wait --for=condition=ready pod --all -n sock-shop --timeout=120s",
    capture=True, check=False   # check=False because some pods may still be pulling images
)

print("\n--- Sock Shop pods (look for 2/2 READY) ---")
run("kubectl get pods -n sock-shop", capture=True)

$ kubectl apply -f sockshop/sock-shop-demo.yaml
namespace/sock-shop created
serviceaccount/front-end created
service/front-end created
deployment.apps/front-end created
serviceaccount/catalogue created
service/catalogue created
deployment.apps/catalogue created
serviceaccount/catalogue-db created
service/catalogue-db created
deployment.apps/catalogue-db created
serviceaccount/carts created
service/carts created
deployment.apps/carts created
serviceaccount/carts-db created
service/carts-db created
deployment.apps/carts-db created
serviceaccount/orders created
service/orders created
deployment.apps/orders created


Waiting for Sock Shop pods to start (this takes 1-2 minutes)...
$ kubectl wait --for=condition=ready pod --all -n sock-shop --timeout=120s
pod/carts-85f899d8cf-g4fr7 condition met
pod/carts-db-787cb57b5-74r8d condition met
pod/catalogue-68c47ffd46-hzqr2 condition met
pod/catalogue-db-67b98f5dcd-2hwdp condition met
pod/front-end-58bd5f4987-h54mj condition met
pod/orders-75f5dff

## Step 6 — Verify SPIFFE Integration

This is the key validation step. We confirm that:

1. Each Istio sidecar has **received a certificate** from Teleport (not from Istio's own CA).
2. The certificate's SAN contains the correct **SPIFFE ID** in the format Istio expects.

`validate-spiffe-ids.sh` queries the Envoy proxy admin API on port 15000 inside each pod, extracts the certificate, and compares the actual SPIFFE SAN against the expected format.

`teleport-cert-demo.sh` decodes the raw certificate and prints the full X.509 details — useful for confirming the issuer is Teleport.

In [57]:
print("=" * 60)
print("Validating SPIFFE IDs on all Sock Shop sidecars")
print("=" * 60)
run("./validate-spiffe-ids.sh", capture=True)

Validating SPIFFE IDs on all Sock Shop sidecars
$ ./validate-spiffe-ids.sh
What this script checks:
  Each sock-shop pod has an Istio sidecar (istio-proxy) holding a SPIFFE certificate.
  In this demo, those certs are issued by Teleport (via tbot) instead of Istio's built-in CA.

  Expected SPIFFE ID: spiffe://<teleport-cluster-domain>/ns/<namespace>/sa/<service-account>
  Expected  = built from TELEPORT_TRUST_DOMAIN in .env + the pod's namespace and service account
  Actual    = extracted from the live Envoy config at localhost:15000/config_dump inside istio-proxy

  A mismatch means the trust domain in the issued cert doesn't match your Teleport cluster.
  Common causes: Istio was installed before configure-trust-domain.sh ran, or tbot is not running.

=== Service: front-end ===
Pod: front-end-58bd5f4987-h54mj
ServiceAccount: front-end
Expected SPIFFE ID: spiffe://ellinj.teleport.sh/ns/sock-shop/sa/front-end
  Running: kubectl exec -n sock-shop front-end-58bd5f4987-h54mj -c istio-pro

In [58]:
print("=" * 60)
print("Inspecting a decoded Teleport-issued certificate")
print("=" * 60)
run("./teleport-cert-demo.sh", capture=True)

Inspecting a decoded Teleport-issued certificate
$ ./teleport-cert-demo.sh
╔════════════════════════════════════════════════════════════════╗
║     TELEPORT WORKLOAD IDENTITY CERTIFICATE ISSUANCE DEMO       ║
╚════════════════════════════════════════════════════════════════╝

📋 Step 1: Show Workload Identity Configuration
─────────────────────────────────────────────────────────────────
This configuration defines the SPIFFE ID template for Istio workloads:

spec:
  spiffe:
    id: /ns/{{ workload.kubernetes.namespace }}/sa/{{ workload.kubernetes.service_account
      }}
version: v1

📦 Step 2: List Active Workloads Getting Certificates
─────────────────────────────────────────────────────────────────
carts-85f899d8cf-g4fr7 - 2/2 containers - Running
carts-db-787cb57b5-74r8d - 2/2 containers - Running
catalogue-68c47ffd46-hzqr2 - 2/2 containers - Running
catalogue-db-67b98f5dcd-2hwdp - 2/2 containers - Running
front-end-58bd5f4987-h54mj - 2/2 containers - Running
orders-75f5dffd8b-sf5zs 

## Step 7 — Test the Application Before Policies

At this point:
- **mTLS is active** — all sidecar-to-sidecar traffic is encrypted and mutually authenticated.
- **No authorization policies exist** — any authenticated service can call any other service.

This is the baseline state that proves the application works end-to-end with Teleport certificates before we lock it down. Both `curl` calls should return HTTP 200.

In [59]:
# Grab the external LoadBalancer IP assigned to the front-end service
result = run(
    "kubectl get svc -n sock-shop front-end -o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    capture=True
)
FRONTEND_IP = result.stdout.strip().strip("'")

if not FRONTEND_IP:
    print("⚠️  No LoadBalancer IP yet — the cloud provider may still be provisioning it.")
    print("   Re-run this cell in ~30 seconds.")
else:
    print(f"Frontend IP: {FRONTEND_IP}")
    
    run("kubectl get svc -n sock-shop front-end", capture=True)

    print("\n--- Test 1: home page (expect HTTP 200) ---")
    run(f"curl -s -o /dev/null -w 'HTTP %{{http_code}}\n' http://{FRONTEND_IP}/", capture=True)

    print("\n--- Test 2: catalogue endpoint (expect HTTP 200) ---")
    run(f"curl -s -o /dev/null -w 'HTTP %{{http_code}}\n' http://{FRONTEND_IP}/catalogue", capture=True)

$ kubectl get svc -n sock-shop front-end -o jsonpath='{.status.loadBalancer.ingress[0].ip}'
192.168.1.230
Frontend IP: 192.168.1.230
$ kubectl get svc -n sock-shop front-end
NAME        TYPE           CLUSTER-IP       EXTERNAL-IP     PORT(S)        AGE
front-end   LoadBalancer   10.107.189.146   192.168.1.230   80:31421/TCP   94s


--- Test 1: home page (expect HTTP 200) ---
$ curl -s -o /dev/null -w 'HTTP %{http_code}
' http://192.168.1.230/
HTTP 200


--- Test 2: catalogue endpoint (expect HTTP 200) ---
$ curl -s -o /dev/null -w 'HTTP %{http_code}
' http://192.168.1.230/catalogue
HTTP 200



## Step 8 — Apply Zero-Trust Policies

### 8a. Break it: deny-all

An Istio `AuthorizationPolicy` with an **empty spec** is a *deny-all* policy — it denies every request to every service in the namespace. The `/catalogue` endpoint should now return `403 Forbidden`.

This demonstrates the zero-trust principle: start with nothing allowed, then explicitly permit what you need.

```yaml
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata:
  name: deny-all
  namespace: sock-shop
spec: {}   # empty = deny everything
```

In [60]:
run("kubectl apply -f sockshop/sock-shop-deny-all.yaml", capture=True)

import time
print("Waiting 5 seconds for policy to propagate to all sidecars...")
time.sleep(5)

print("\n--- Test: catalogue should now FAIL (403 or connection refused) ---")
run(f"curl -s -o /dev/null -w 'HTTP %{{http_code}}\n' http://{FRONTEND_IP}/catalogue", capture=True, check=False)

$ kubectl apply -f sockshop/sock-shop-deny-all.yaml
authorizationpolicy.security.istio.io/deny-all created

Waiting 5 seconds for policy to propagate to all sidecars...

--- Test: catalogue should now FAIL (403 or connection refused) ---
$ curl -s -o /dev/null -w 'HTTP %{http_code}
' http://192.168.1.230/catalogue
HTTP 403



### 8b. Fix it: SPIFFE-based allow policies

`sock-shop-policies.yaml` contains the full set of Istio security resources:

| Resource | Purpose |
|----------|---------|
| `AuthorizationPolicy` per service | Defines which **SPIFFE IDs** (callers) are allowed to reach each service, and which HTTP methods/paths are permitted |
| `PeerAuthentication` (STRICT) | Enforces that all inbound connections must present a valid mTLS certificate — no plaintext |
| `PeerAuthentication` (PERMISSIVE on front-end) | Allows external browsers (which can't present Teleport certs) to reach the front-end |
| `DestinationRule` for databases | Tells the *client* sidecar to use mTLS when connecting to the database services |

Example allow policy:
```yaml
spec:
  selector:
    matchLabels: { app: catalogue }
  action: ALLOW
  rules:
  - from:
    - source:
        principals:
        - "YOUR-DOMAIN/ns/sock-shop/sa/front-end"   # only front-end can call catalogue
    to:
    - operation:
        methods: ["GET"]
        paths: ["/catalogue*", "/tags", "/health"]
```

Note: Istio automatically prepends `spiffe://` to the principal, so the YAML only needs `DOMAIN/ns/.../sa/...`.

In [61]:
run("kubectl apply -f sockshop/sock-shop-policies.yaml", capture=True)

print("\nWaiting 5 seconds for policies to propagate...")
time.sleep(5)

print("\n--- Test: catalogue should WORK again (200) ---")
run(f"curl -s -o /dev/null -w 'HTTP %{{http_code}}\n' http://{FRONTEND_IP}/catalogue", capture=True, check=False)

print("\n--- Test: home page should WORK (200) ---")
run(f"curl -s -o /dev/null -w 'HTTP %{{http_code}}\n' http://{FRONTEND_IP}/", capture=True, check=False)

$ kubectl apply -f sockshop/sock-shop-policies.yaml
authorizationpolicy.security.istio.io/deny-all unchanged
authorizationpolicy.security.istio.io/catalogue-allow-frontend created
authorizationpolicy.security.istio.io/catalogue-db-allow-catalogue created
authorizationpolicy.security.istio.io/carts-db-allow-carts created
authorizationpolicy.security.istio.io/carts-allow-frontend created
authorizationpolicy.security.istio.io/orders-allow-frontend created
authorizationpolicy.security.istio.io/frontend-allow-external created
peerauthentication.security.istio.io/default created
peerauthentication.security.istio.io/catalogue-db-strict created
peerauthentication.security.istio.io/carts-db-mtls created
peerauthentication.security.istio.io/frontend-permissive created
destinationrule.networking.istio.io/catalogue-db created
destinationrule.networking.istio.io/carts-db created


Waiting 5 seconds for policies to propagate...

--- Test: catalogue should WORK again (200) ---
$ curl -s -o /dev/null 

## Step 9 — Verify mTLS and Authorization Enforcement

### 9a. Confirm mTLS is active on the wire

Envoy (the proxy that backs every Istio sidecar) exposes a statistics endpoint on port 15000. The metric `connection_security_policy.mutual_tls` counts connections that were authenticated using mTLS.

A non-zero counter here proves that traffic between services is **encrypted and mutually authenticated** using the Teleport-issued SPIFFE certificates.

In [62]:
import re

# Fetch stats from every pod in the namespace, but keep only destination-reporter lines.
# The destination sidecar correctly labels mTLS even for TCP — avoiding the source-reporter
# limitation that shows 'unknown' for outbound TCP connections.
pods = subprocess.run(
    "kubectl get pods -n sock-shop -o jsonpath='{.items[*].metadata.name}'",
    shell=True, capture_output=True, text=True, cwd=REPO_DIR
).stdout.strip().strip("'").split()

print(f"Querying {len(pods)} pods: {', '.join(pods)}\n")

SPIFFE_SRC  = re.compile(r"source_principal\.(spiffe://[^/]+/ns/[^/]+/sa/[^\s.]+)")
SPIFFE_DST  = re.compile(r"destination_principal\.(spiffe://[^/]+/ns/[^/]+/sa/[^\s.]+)")
METRIC_NAME = re.compile(r"^istiocustom\.(\w+)\.")

rows = []
for pod in pods:
    raw = subprocess.run(
        f"kubectl exec -n sock-shop {pod} -c istio-proxy -- curl -s localhost:15000/stats",
        shell=True, capture_output=True, text=True, cwd=REPO_DIR
    ).stdout
    for line in raw.splitlines():
        if "reporter.destination" not in line:
            continue
        if "connection_security_policy.mutual_tls" not in line:
            continue
        metric_str, _, value_str = line.rpartition(": ")
        if not value_str.strip().isdigit():
            continue
        count = int(value_str.strip())
        if count == 0:
            continue
        src_m  = SPIFFE_SRC.search(metric_str)
        dst_m  = SPIFFE_DST.search(metric_str)
        kind_m = METRIC_NAME.match(metric_str)
        if not (src_m and dst_m):
            continue
        rows.append((
            src_m.group(1).split("/sa/", 1)[1],
            dst_m.group(1).split("/sa/", 1)[1],
            kind_m.group(1) if kind_m else "?",
            count,
        ))

if rows:
    w = [max(len(r[i]) for r in rows) for i in range(3)]
    w = [max(w[0], 4), max(w[1], 2), max(w[2], 6)]
    print(f"  {'from':<{w[0]}}  →  {'to':<{w[1]}}  {'metric':<{w[2]}}  {'count':>6}")
    print("  " + "─" * (w[0] + w[1] + w[2] + 22))
    for src, dst, metric, count in rows:
        print(f"  {src:<{w[0]}}  →  {dst:<{w[1]}}  {metric:<{w[2]}}  {count:>6}")
else:
    print("No mutual_tls traffic observed yet — try browsing the front-end and re-running.")

Querying 6 pods: carts-85f899d8cf-g4fr7, carts-db-787cb57b5-74r8d, catalogue-68c47ffd46-hzqr2, catalogue-db-67b98f5dcd-2hwdp, front-end-58bd5f4987-h54mj, orders-75f5dffd8b-sf5zs

  from       →  to            metric                               count
  ─────────────────────────────────────────────────────────────────────────────
  carts      →  carts-db      istio_tcp_connections_closed_total       1
  carts      →  carts-db      istio_tcp_connections_opened_total       2
  carts      →  carts-db      istio_tcp_received_bytes_total         938
  carts      →  carts-db      istio_tcp_sent_bytes_total            7780
  front-end  →  catalogue     istio_requests_total                     2
  catalogue  →  catalogue-db  istio_tcp_connections_opened_total       1
  catalogue  →  catalogue-db  istio_tcp_received_bytes_total         744
  catalogue  →  catalogue-db  istio_tcp_sent_bytes_total            6304


### 9b. Confirm unauthorized access is blocked

We launch a plain `curl` pod **without a proper SPIFFE identity** (no ServiceAccount mapped to an allowed principal in the policy). When it tries to call the `catalogue` service:

1. mTLS handshake succeeds — the pod's sidecar does get a Teleport-issued cert.
2. The `AuthorizationPolicy` check **fails** — the calling SPIFFE ID (`default` ServiceAccount) is not in the allow list for `catalogue`.
3. Envoy returns `403 RBAC: access denied`.

This is zero-trust in action: encryption alone isn't enough — the caller's identity must be explicitly authorized.

In [63]:
# Clean up any leftover test pod from a previous run
run("kubectl delete pod test-curl -n sock-shop --ignore-not-found", capture=True)

# Launch a pod with no special ServiceAccount (uses 'default')
run(
    "kubectl run test-curl -n sock-shop --image=curlimages/curl:latest --restart=Never -- "
    "sh -c 'curl -v http://catalogue/catalogue 2>&1; sleep 5'",
    capture=True
)

print("Waiting for test pod to attempt the call...")
time.sleep(8)

print("\n--- Logs from test pod (expect RBAC denied / 403) ---")
run("kubectl logs test-curl -n sock-shop", capture=True, check=False)

run("kubectl delete pod test-curl -n sock-shop", capture=True)

$ kubectl delete pod test-curl -n sock-shop --ignore-not-found
$ kubectl run test-curl -n sock-shop --image=curlimages/curl:latest --restart=Never -- sh -c 'curl -v http://catalogue/catalogue 2>&1; sleep 5'
pod/test-curl created

Waiting for test pod to attempt the call...

--- Logs from test pod (expect RBAC denied / 403) ---
$ kubectl logs test-curl -n sock-shop
* Host catalogue:80 was resolved.
* IPv6: (none)
* IPv4: 10.99.75.33
*   Trying 10.99.75.33:80...
* connect to 10.99.75.33 port 80 from 10.244.230.44 port 54262 failed: Connection refused
* Failed to connect to catalogue port 80 after 6 ms: Could not connect to server
* closing connection #0
curl: (7) Failed to connect to catalogue port 80 after 6 ms: Could not connect to server

$ kubectl delete pod test-curl -n sock-shop
pod "test-curl" deleted from sock-shop namespace



## Step 10 — Architecture Summary

What just happened, end-to-end:

```
┌─────────────────────────────────────────────────────────┐
│  Kubernetes Node                                        │
│                                                         │
│  ┌───────────────┐   SPIFFE       ┌──────────────────┐  │
│  │  tbot pod     │◀──Workload API─│  Istio sidecar   │  │
│  │  (DaemonSet)  │──SVID cert──▶  │  (istio-proxy)   │  │
│  └───────┬───────┘  /run/spire/   └────────┬─────────┘  │
│          │          sockets/socket          │            │
│          │ joins via k8s token              │ mTLS       │
│          ▼                                  ▼            │
│  ┌───────────────┐                ┌──────────────────┐  │
│  │  Teleport     │                │  App container   │  │
│  │  cluster      │                │  (catalogue,     │  │
│  │               │                │   front-end, …)  │  │
│  └───────────────┘                └──────────────────┘  │
└─────────────────────────────────────────────────────────┘
```

**Data flow for a service-to-service call:**
1. `front-end` calls `catalogue` → intercepted by front-end's Envoy sidecar.
2. Envoy presents its **Teleport SPIFFE cert** (`spiffe://…/ns/sock-shop/sa/front-end`).
3. `catalogue`'s Envoy verifies the cert against the mesh trust domain.
4. Envoy checks the matching `AuthorizationPolicy` — principal `…/sa/front-end` on path `/catalogue*` is ALLOW.
5. Request is forwarded to the catalogue app container.

**Why this matters over plain Istio:**
- Certificates are issued by **Teleport's CA**, not a cluster-local CA. Revocation and audit happen in Teleport.
- The same SPIFFE identity can be used to grant **Teleport access** (SSH, database, app) — one identity for mesh *and* infrastructure.
- Short TTLs (4 min cert lifetime, renewed every 2 min) minimize the blast radius of a compromised cert.

## Step 11 — Cleanup

`cleanup.sh` removes everything created by this demo:
- `istio-system` namespace (Istio)
- `teleport-system` namespace (tbot)
- `sock-shop` namespace (demo application)
- Teleport resources: token, bot role, workload identity

Run this cell when you're done to avoid leaving orphaned resources in your cluster.

In [ ]:
# Uncomment the line below to run cleanup
run("./cleanup.sh", capture=True)
print("Cleanup skipped. Uncomment the run() call above when ready.")

## Step 12 — Export Executed Notebook as HTML

Run this cell after a successful demo run to save a static HTML snapshot of all outputs.
The HTML file opens in any browser with no Jupyter, Python, or cluster access required —
use it as a fallback if the live environment is unavailable during a demo.

In [ ]:
import sys
.env_path
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
full_file  = f"demo-walkthrough-executed-{timestamp}.html"

run(
    f"{sys.executable} -m jupyter nbconvert --to html"
    f" demo-walkthrough.ipynb --output {full_file}",
    capture=True
)
print(f"✅ Full (with code):     {os.path.join(REPO_DIR, full_file)}")
